In [ ]:
"""
load_traffic_data.py
====================
NJ TMAS Traffic Data Pipeline
Loads all 15 VOL files, computes Average Daily Traffic (ADT) per station
per season, joins with GPS coordinates from the STA metadata file, and
saves a clean master CSV ready for mapping.

Output: data/nj_traffic_adt_master.csv
"""

import os
import pandas as pd

# CONFIG
UPLOADS_DIR = "data/Traffic"   # where VOL and STA files live
OUTPUT_DIR  = "data"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "nj_traffic_adt_master.csv")

# Summer = tourism season (Jun–Aug); Winter = clean baseline (Nov–Dec)
SUMMER_MONTHS = {6, 7, 8}
WINTER_MONTHS = {11, 12}

# The 24 hourly count columns in every VOL row
HOUR_COLS = [f"hour_{str(h).zfill(2)}" for h in range(24)]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# STEP 1: LOAD ALL VOL FILES
print("Loading VOL files...")

vol_files = sorted(
    f for f in os.listdir(UPLOADS_DIR) if f.endswith(".VOL")
)
print(f"  Found {len(vol_files)} VOL files: {vol_files}")

daily_records = []

for fname in vol_files:
    path = os.path.join(UPLOADS_DIR, fname)
    df = pd.read_csv(path, sep="|", low_memory=False)
    df.columns = df.columns.str.lower().str.strip()

    # Keep only summer and winter months
    month = df["month_record"].iloc[0]
    if month not in SUMMER_MONTHS | WINTER_MONTHS:
        print(f"  Skipping {fname} — month {month} not in target seasons")
        continue

    # Sum all 24 hourly columns to get total vehicles per lane+direction+day
    df["daily_total"] = df[HOUR_COLS].sum(axis=1)

    # Collapse lanes & directions → one row per station per calendar day
    # This gives total bidirectional volume through the station
    daily = (
        df.groupby(["station_id", "year_record", "month_record", "day_record"])
        ["daily_total"]
        .sum()
        .reset_index()
    )
    daily_records.append(daily)
    print(f"  Loaded {fname}: {len(daily):,} station-days")

combined = pd.concat(daily_records, ignore_index=True)
print(f"\nCombined: {len(combined):,} station-day rows across {combined['station_id'].nunique()} stations")

# STEP 2: TAG SEASONS AND COMPUTE ADT
combined["season"] = combined["month_record"].apply(
    lambda m: "summer" if m in SUMMER_MONTHS else "winter"
)

# Average Daily Traffic = mean of all daily_total values in that season
# Multi-year averaging reduces single-year anomalies
summer_adt = (
    combined[combined["season"] == "summer"]
    .groupby("station_id")["daily_total"]
    .mean()
    .rename("summer_adt")
)
winter_adt = (
    combined[combined["season"] == "winter"]
    .groupby("station_id")["daily_total"]
    .mean()
    .rename("winter_adt")
)

adt_df = pd.concat([summer_adt, winter_adt], axis=1).reset_index()
adt_df["pct_change"] = (
    (adt_df["summer_adt"] - adt_df["winter_adt"]) / adt_df["winter_adt"] * 100
).round(1)

print(f"\nADT computed for {len(adt_df)} stations")

# STEP 3: JOIN WITH GPS COORDINATES FROM STA FILE
print("\nLoading STA metadata...")
sta_path = os.path.join(UPLOADS_DIR, "NJ_2025_(TMAS).STA")
sta = pd.read_csv(sta_path, sep="|")
sta.columns = sta.columns.str.lower().str.strip()

# Deduplicate on station_id — STA has one row per lane/direction combination
# Coordinates are identical for all lanes at the same station, so first() is fine
sta_dedup = (
    sta[["station_id", "latitude", "longitude", "station_location", "county_code"]]
    .drop_duplicates("station_id")
    .copy()
)
print(f"  STA: {len(sta_dedup)} unique stations")

# STEP 4: MERGE AND CLEAN
merged = adt_df.merge(sta_dedup, on="station_id", how="left")

# Drop stations without coordinates (can't be mapped)
before = len(merged)
merged = merged.dropna(subset=["latitude", "longitude", "summer_adt"])
print(f"  Dropped {before - len(merged)} stations missing coords or ADT")
print(f"  Final mapped stations: {len(merged)}")

# Round for readability (some stations only have summer or winter data — keep as float)
merged["summer_adt"] = merged["summer_adt"].round(0)
merged["winter_adt"] = merged["winter_adt"].round(0)

# STEP 5: ZONE LABELING
# Assign project zones based on county_code (NJ FIPS county codes)
# Bergen=3, Hudson=9, Passaic=31, Essex=13 → Corridor
# Atlantic=1, Monmouth=25, Ocean=29 → Shore
# All others → Statewide

CORRIDOR_COUNTIES = {3, 9, 31}    # Bergen, Hudson, Passaic
SHORE_COUNTIES    = {1, 25, 29}   # Atlantic, Monmouth, Ocean
ESSEX_COUNTY      = {13}

def assign_zone(county_code):
    try:
        c = int(county_code)
    except (ValueError, TypeError):
        return "Statewide"
    if c in CORRIDOR_COUNTIES:
        return "Transit Corridor"
    if c in SHORE_COUNTIES:
        return "Jersey Shore"
    if c in ESSEX_COUNTY:
        return "Newark Gateway"
    return "Statewide"

merged["zone"] = merged["county_code"].apply(assign_zone)
print("\nZone distribution:")
print(merged["zone"].value_counts().to_string())

# STEP 6: SAVE
merged.to_csv(OUTPUT_FILE, index=False)
print(f"\n✓ Saved: {OUTPUT_FILE}")
print(f"  Columns: {list(merged.columns)}")
print("\nTop 10 stations by summer ADT:")
print(
    merged.nlargest(10, "summer_adt")[
        ["station_id", "station_location", "summer_adt", "winter_adt", "pct_change", "zone"]
    ].to_string(index=False)
)

Loading VOL files...
  Found 15 VOL files: ['NJ_AUG_2023 (TMAS).VOL', 'NJ_AUG_2024 (TMAS).VOL', 'NJ_Aug_2025 (TMAS).VOL', 'NJ_DEC_2023 (TMAS).VOL', 'NJ_DEC_2024 (TMAS).VOL', 'NJ_Dec_2025 (TMAS).VOL', 'NJ_JUL_2023 (TMAS).VOL', 'NJ_JUL_2024 (TMAS).VOL', 'NJ_JUN_2023 (TMAS).VOL', 'NJ_JUN_2024 (TMAS).VOL', 'NJ_Jul_2025 (TMAS).VOL', 'NJ_Jun_2025 (TMAS).VOL', 'NJ_NOV_2023 (TMAS).VOL', 'NJ_NOV_2024 (TMAS).VOL', 'NJ_Nov_2025 (TMAS).VOL']
  Loaded NJ_AUG_2023 (TMAS).VOL: 5,024 station-days
  Loaded NJ_AUG_2024 (TMAS).VOL: 4,446 station-days
  Loaded NJ_Aug_2025 (TMAS).VOL: 3,835 station-days
  Loaded NJ_DEC_2023 (TMAS).VOL: 5,008 station-days
  Loaded NJ_DEC_2024 (TMAS).VOL: 4,594 station-days
  Loaded NJ_Dec_2025 (TMAS).VOL: 4,121 station-days
  Loaded NJ_JUL_2023 (TMAS).VOL: 4,908 station-days
  Loaded NJ_JUL_2024 (TMAS).VOL: 4,543 station-days
  Loaded NJ_JUN_2023 (TMAS).VOL: 4,768 station-days
  Loaded NJ_JUN_2024 (TMAS).VOL: 4,605 station-days
  Loaded NJ_Jul_2025 (TMAS).VOL: 3,914 station